<a href="https://colab.research.google.com/github/TlalaneLepolesa/API-MONGO-DB/blob/main/AI_Group_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('/content/drugsComTest_raw.csv')

# 2. See how many rows and columns we are starting with
print(f"Original data size: {df.shape}")

# 3. Filter the data to ONLY include the three conditions required by the assignment
target_conditions = ['Depression', 'High Blood Pressure', 'Type 2 Diabetes']
filtered_df = df[df['condition'].isin(target_conditions)]

# 4. Check the new size of our data
print(f"Filtered data size: {filtered_df.shape}")

# 5. Display the first 5 rows to see what it looks like
filtered_df.head()

Original data size: (53766, 7)
Filtered data size: (3878, 7)


,uniqueID,drugName,condition,review,rating,date,usefulCount
0,163740,Mirtazapine,Depression,"""I&#039;ve tried a few antidepressants over th...",10,28-Feb-12,22
38,141462,Escitalopram,Depression,"""I am a 22 year old female college student. I ...",9,29-Apr-14,32
67,201582,Zoloft,Depression,"""Zoloft did not help me at all. I was on it f...",1,14-Jan-13,51
73,131683,Effexor XR,Depression,"""Sadly only lasted 5 days on Effexor XR. The s...",1,24-Apr-16,18
79,122089,Venlafaxine,Depression,"""I was first prescribed Effexor 13 years ago a...",8,13-Dec-10,36


In [9]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. FIX THE CONDITIONS (Include the exact spelling for Diabetes)
target_conditions = ['Depression', 'High Blood Pressure', 'Diabetes, Type 2']
# We use .copy() so we don't mess up your original data
filtered_df = df[df['condition'].isin(target_conditions)].copy()

print(f"Corrected Filtered data size: {filtered_df.shape} rows")

# 2. THE CLEANING FUNCTION
def clean_text(text):
    text = str(text) # Make sure it reads as text
    text = re.sub(r'&#\d+;', '', text) # Remove HTML junk like &#039;
    text = text.lower() # Lowercase everything
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation and numbers
    return text

# Apply the cleaning to your reviews
filtered_df['clean_review'] = filtered_df['review'].apply(clean_text)

# 3. SPLITTING THE DATA (Study Set vs. Exam Set)
X = filtered_df['clean_review']
y = filtered_df['condition']
# Hide 20% for the final exam
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. VECTORIZATION (Turning words into numbers using TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. TRAINING THE AI MODEL (The "Brain")
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. TEST THE AI
y_pred = model.predict(X_test_tfidf)

# Show the results!
print(f"\n--- AI Results ---")
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nDetailed Performance Report:")
print(classification_report(y_test, y_pred))

Corrected Filtered data size: (4686, 7) rows

--- AI Results ---
Model Accuracy: 86.89%

Detailed Performance Report:
                     precision    recall  f1-score   support

         Depression       0.84      1.00      0.91       631
   Diabetes, Type 2       1.00      0.64      0.78       158
High Blood Pressure       1.00      0.56      0.72       149

           accuracy                           0.87       938
          macro avg       0.95      0.73      0.80       938
       weighted avg       0.89      0.87      0.86       938



In [10]:
import re

def clean_review(text):
    # 1. Remove HTML codes like &#039;
    text = re.sub(r'&#\d+;', '', text)
    # 2. Lowercase everything (so "Depression" and "depression" are the same)
    text = text.lower()
    # 3. Remove punctuation and numbers (we just want the words)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# Apply the cleaning to a new column called 'clean_review'
filtered_df['clean_review'] = filtered_df['review'].apply(clean_review)

# See the difference!
filtered_df[['review', 'clean_review']].head()

,review,clean_review
0,"""I&#039;ve tried a few antidepressants over th...",ive tried a few antidepressants over the years...
35,"""Have been on Actos for almost a year, gained ...",have been on actos for almost a year gained p...
38,"""I am a 22 year old female college student. I ...",i am a year old female college student i want...
67,"""Zoloft did not help me at all. I was on it f...",zoloft did not help me at all i was on it for...
73,"""Sadly only lasted 5 days on Effexor XR. The s...",sadly only lasted days on effexor xr the side...


In [11]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. THE CLEANING FUNCTION
def clean_text(text):
    # Remove HTML junk like &#039;
    text = re.sub(r'&#\d+;', '', text)
    # Lowercase everything
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    return text

# Apply the cleaning to your reviews
filtered_df['clean_review'] = filtered_df['review'].apply(clean_text)

# 2. SPLITTING THE DATA (Study Set vs. Exam Set)
# We use 80% to train (study) and 20% to test (the final exam)
X = filtered_df['clean_review']
y = filtered_df['condition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. VECTORIZATION (Turning words into numbers)
# We use TF-IDF to find "important" words and ignore common ones like 'the'
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 4. TRAINING THE AI MODEL (The "Brain")
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 5. TEST THE AI
y_pred = model.predict(X_test_tfidf)

# Show the results
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\n--- Detailed Performance Report ---")
print(classification_report(y_test, y_pred))

Model Accuracy: 86.89%

--- Detailed Performance Report ---
                     precision    recall  f1-score   support

         Depression       0.84      1.00      0.91       631
   Diabetes, Type 2       1.00      0.64      0.78       158
High Blood Pressure       1.00      0.56      0.72       149

           accuracy                           0.87       938
          macro avg       0.95      0.73      0.80       938
       weighted avg       0.89      0.87      0.86       938

